In [ ]:
import os
import numpy as np

import mlflow
from mlflow.tracking import MlflowClient

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


In [ ]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("diabets-exp")

In [ ]:
data = load_diabetes(scaled=False)
X, y = data.data, data.target
feature_names = data.feature_names
feature_names


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipe = make_pipeline(
    StandardScaler(),
    RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    )
)

with mlflow.start_run() as run:
    mlflow.log_param("model_type", "RandomForestRegressor")
    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("random_state", 42)

    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    rmse = mean_squared_error(y_test, pred, squared=False)
    r2 = r2_score(y_test, pred)

    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    model_info = mlflow.sklearn.log_model(
        sk_model=pipe,
        artifact_path="model",
        registered_model_name="diabets",
        input_example=X_test[:2],
    )

    run_id = run.info.run_id

run_id, rmse, r2, model_info.model_uri


In [ ]:
client = MlflowClient()
versions = client.search_model_versions("name='diabets'")

latest = sorted(versions, key=lambda v: int(v.version))[-1]
latest.version, latest.current_stage, latest.status, latest.run_id


In [ ]:
version = latest.version

model = mlflow.sklearn.load_model(f"models:/diabets/{version}")



In [ ]:
import shutil
from pathlib import Path
export_path = Path("..") / "mlapp" / "model"

if export_path.exists():
    shutil.rmtree(export_path)

mlflow.sklearn.save_model(model, path=str(export_path))

export_path, list(export_path.iterdir())[:5]